# 03_gbdt

勾配ブースティング（sklearn の HistGradientBoostingRegressor）を，02 と同じ特徴量・同じ CV で動かして，ランダムフォレストと比べる．設定は既定のまま．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import PredefinedSplit, cross_val_predict, cross_val_score

from bike_sharing.cv import get_folds
from bike_sharing.data import load_data, save_submission
from bike_sharing.visualize import set_style

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
set_style()

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

## 特徴量と目的変数

02 と同じ．

In [ ]:
for df in [train, test]:
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["hour"] = df["datetime"].dt.hour
    df["dow"] = df["datetime"].dt.dayofweek

features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed",
            "year", "month", "hour", "dow"]
X = train[features]
X_test = test[features]
y = np.log1p(train["count"])

block_cv = PredefinedSplit(get_folds(train))

## CV スコア

In [ ]:
# 02 と同じランダムフォレスト
rf = RandomForestRegressor(random_state=0)
rf_scores = -cross_val_score(rf, X, y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("ランダムフォレスト:", rf_scores.round(4), " 平均", rf_scores.mean().round(4))

In [ ]:
# 既定の設定のまま．ただし早期終了だけは切る
# 既定（auto）だと 10000 行を超えたときだけ早期終了が入るので，CV の各 fold（約 8000 行）では入らず，
# 最後に train 全体（10886 行）で学習するときだけ入ってしまう
gb = HistGradientBoostingRegressor(early_stopping=False)
gb_scores = -cross_val_score(gb, X, y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("勾配ブースティング:", gb_scores.round(4), " 平均", gb_scores.mean().round(4))

In [ ]:
result = pd.DataFrame({
    "ランダムフォレスト": rf_scores,
    "勾配ブースティング": gb_scores,
}, index=["1〜5 日", "6〜10 日", "11〜14 日", "15〜19 日"])
result.loc["平均"] = result.mean()
result["差"] = result["勾配ブースティング"] - result["ランダムフォレスト"]
result.round(4)

## どこで差が出たか

02 と同じく，各行が検証側になったときの予測（out-of-fold 予測）で比べる．

In [ ]:
oof_rf = cross_val_predict(rf, X, y, cv=block_cv)
oof_gb = cross_val_predict(gb, X, y, cv=block_cv)
print("ランダムフォレスト:", round(root_mean_squared_error(y, oof_rf), 4))
print("勾配ブースティング:", round(root_mean_squared_error(y, oof_gb), 4))

train["pred_rf"] = np.expm1(oof_rf)
train["pred_gb"] = np.expm1(oof_gb)

In [ ]:
# 時間ごとの RMSLE
err = pd.DataFrame({
    "hour": train["hour"],
    "ランダムフォレスト": (oof_rf - y) ** 2,
    "勾配ブースティング": (oof_gb - y) ** 2,
})
err_hour = np.sqrt(err.groupby("hour").mean())

plt.figure(figsize=(8, 4))
plt.plot(err_hour.index, err_hour["ランダムフォレスト"], marker="o", label="ランダムフォレスト")
plt.plot(err_hour.index, err_hour["勾配ブースティング"], marker="o", label="勾配ブースティング")
plt.title("時間ごとの RMSLE")
plt.xlabel("時間")
plt.legend()
plt.show()

In [ ]:
# 時間ごとの平均台数を，勤務日と土日祝に分けて比べる
wd = train[train["workingday"] == 1].groupby("hour")[["count", "pred_rf", "pred_gb"]].mean()
hd = train[train["workingday"] == 0].groupby("hour")[["count", "pred_rf", "pred_gb"]].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ax[0].plot(wd.index, wd["count"], marker=".", label="実際")
ax[0].plot(wd.index, wd["pred_rf"], marker=".", label="ランダムフォレスト")
ax[0].plot(wd.index, wd["pred_gb"], marker=".", label="勾配ブースティング")
ax[1].plot(hd.index, hd["count"], marker=".", label="実際")
ax[1].plot(hd.index, hd["pred_rf"], marker=".", label="ランダムフォレスト")
ax[1].plot(hd.index, hd["pred_gb"], marker=".", label="勾配ブースティング")
ax[0].set_title("勤務日")
ax[1].set_title("土日祝")
ax[0].set_xlabel("時間")
ax[1].set_xlabel("時間")
ax[0].set_ylabel("平均台数")
ax[0].legend()
plt.show()

In [ ]:
# 天気ごと，勤務日かどうかごとの RMSLE（weather = 4 は 1 行しかない）
err["weather"] = train["weather"]
err["workingday"] = train["workingday"]
print(np.sqrt(err.groupby("weather")[["ランダムフォレスト", "勾配ブースティング"]].mean()).round(3))
print()
print(np.sqrt(err.groupby("workingday")[["ランダムフォレスト", "勾配ブースティング"]].mean()).round(3))

## 提出用のファイル

作っておくだけで，提出は Kaggle の画面からする．

In [ ]:
# 全部の train で学習し直して test を予測する
gb.fit(X, y)
pred_gb = np.expm1(gb.predict(X_test))
print("0 未満になった行:", (pred_gb < 0).sum())

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_gb}), "03_gbdt")